# Gender Wage Gap in the US, 1976-2025
CPS ASEC from IPUMS | Course research project

**Question.** How much has the gender wage gap closed in the US since 1976, and for whom?

**Rule:** every table and chart uses the survey weight, `asecwt`. Unweighted counts are only for checking data.

**Table first, chart second.** Print the weighted table (1 decimal), then chart exactly that table. Bar and line charts start at 0.

---
## 0. Setup
The file `jobs_assignment_data_20261104.csv` is in the root of My Drive: income years 1976-2025, ages 16-64, wage and salary workers with positive wages. The file has 85 columns and is 1.6 GB, so we load only the 18 below. `year` is the **income year** (the year the wages were earned). The setup cell adds a text label for each coded variable (`female_label`, `fulltime_label`, `educ5_label`, and so on).

| Variable | Meaning |
|---|---|
| `year`, `age`, `asecwt` | income year; age 16-64; weight (Americans each row stands for) |
| `female` | 1 = female, 0 = male |
| `fulltime`, `hrswk`, `wkswork` | full-time (1) or part-time (0) last year, full-time = 35+ usual weekly hours; usual weekly hours; weeks worked last year |
| `incwage_real`, `hrwage_real` | yearly wage and salary income; hourly wage (yearly wage / (weeks x hours)); both in 2025 dollars |
| `agegrp` | 16-24, 25-34, 35-44, 45-54, 55-64 |
| `educ5` | None/preschool, Less than HS, High school, Some college, Bachelor's+ |
| `raceeth` | White, Black (non-Hispanic), Hispanic (any race), Other; missing = don't know or no response |
| `marst3` | Married, Previously married, Never married |
| `nchild_cat`, `child_u5` | number of children (0, 1, 2, 3+); 1 = child under 5 |
| `occ6` | 6 occupation groups; missing = not in universe, military, 999, or other |
| `classwkr3` | Private, Government (the self-employed were removed) |
| `metro` | 1 = in a metro area, 0 = not; missing in some years |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from google.colab import drive
drive.mount("/content/drive")
pd.set_option("display.width", 200)                                             # wide tables stay on one line
pd.set_option("display.max_columns", 30)                                        # show every column

data_path = "/content/drive/My Drive/jobs_assignment_data_20261104.csv"        # change here if the file moves
cols = ["year", "age", "asecwt", "female", "fulltime", "hrswk", "wkswork", "incwage_real", "hrwage_real",
        "agegrp", "educ5", "raceeth", "marst3", "nchild_cat", "child_u5", "occ6", "classwkr3", "metro"]

df = pd.read_csv(data_path, usecols=cols)                                      # read only these columns: the file is big
print(df.shape)                                                                # rows, columns we loaded
print(list(df.columns))                                                        # their names
print(len(pd.read_csv(data_path, nrows=0).columns), "columns in the file")     # nrows=0 reads only the header

In [ ]:
# Labels for the codes, then a check: every code should land in exactly one label
df["female_label"] = df["female"].map({0: "Male", 1: "Female"})
df["fulltime_label"] = df["fulltime"].map({0: "Part-time", 1: "Full-time"})
df["agegrp_label"] = df["agegrp"].map({1: "16-24", 2: "25-34", 3: "35-44", 4: "45-54", 5: "55-64"})
df["educ5_label"] = df["educ5"].map({1: "None/preschool", 2: "Less than HS", 3: "High school", 4: "Some college", 5: "Bachelor's+"})
df["raceeth_label"] = df["raceeth"].map({1: "Hispanic", 2: "White", 3: "Black", 4: "Other"})        # missing stays missing
df["marst3_label"] = df["marst3"].map({1: "Married", 2: "Previously married", 3: "Never married"})
df["nchild_cat_label"] = df["nchild_cat"].map({0: "0", 1: "1", 2: "2", 3: "3+"})
df["child_u5_label"] = df["child_u5"].map({0: "No child under 5", 1: "Child under 5"})
df["occ6_label"] = df["occ6"].map({1: "Managers and professionals", 2: "Technical, sales and admin", 3: "Service",
                                   4: "Farming, forestry and fishing", 5: "Craft and repair", 6: "Operators and laborers"})   # missing stays missing
df["classwkr3_label"] = df["classwkr3"].map({1: "Private", 2: "Government"})
df["metro_label"] = df["metro"].map({0: "Non-metro", 1: "Metro"})                                  # missing stays missing
for name in ["female", "fulltime", "agegrp", "educ5", "raceeth", "marst3", "nchild_cat", "child_u5", "occ6", "classwkr3", "metro"]:
    print(pd.crosstab(df[name], df[name + "_label"], dropna=False))                                # each row should have one non-zero cell

In [ ]:
# Weighted mean: multiply each value by its weight, add up, divide by the sum of the weights
def wmean(data, var):                                                   # a table and a column name
    ok = data[var].notna()                                              # rows that have this column
    return np.average(data.loc[ok, var], weights=data.loc[ok, "asecwt"])   # the weighted mean

---
## 1. Summary table
**Question.** What does each loaded column look like before any math? The mean is weighted with `asecwt`. The count (n), the missing count, the min and the max are unweighted: they are data checks. A mean of a 0/1 column like `female` is a share. A mean of a code with several groups like `educ5` means nothing by itself; we keep the row to check n, missing, min and max.

In [ ]:
summary = pd.DataFrame({
    "n": df[cols].count(),                                  # people with a value
    "missing": df[cols].isna().sum(),                       # people without a value
    "weighted mean": [wmean(df, c) for c in cols],          # mean weighted with asecwt
    "min": df[cols].min(),                                  # smallest value
    "max": df[cols].max(),                                  # largest value
}, index=cols)
print(summary.round(1))

---
## 2. A new tool: weighted percentile
A percentile is the wage that a given share of workers earn less than. With weights, each row stands for `asecwt` people. Sort the rows by wage, add up the weights, and take the first wage where the running total reaches that share of all people. Four made-up people, so you can check by hand: the weights add up to 600, so the running shares are 17%, 33%, 83% and 100%.

In [ ]:
# Weighted percentile: sort by value, add up the weights, take the first value where the running share reaches q
def wquantile(data, var, q):                               # a table, a column name, and q between 0 and 1
    ok = data[var].notna()                                 # rows that have this column
    values = data.loc[ok, var].to_numpy()                  # the values
    weights = data.loc[ok, "asecwt"].to_numpy()            # their weights
    order = np.argsort(values)                             # positions that sort the values
    running = np.cumsum(weights[order]) / weights.sum()    # share of people at or below each value
    return values[order][np.searchsorted(running, q)]      # first value where the running share reaches q

example_df = pd.DataFrame({"wage": [10, 20, 30, 40], "asecwt": [100, 100, 300, 100]})   # 600 people
print(wquantile(example_df, "wage", 0.01), wquantile(example_df, "wage", 0.5), wquantile(example_df, "wage", 0.99))   # 1st = 10, median = 30, 99th = 40

---
## 3. Trim cutoffs
**Question.** Which hourly wages are too extreme to use? `hrwage_real` runs from about $0.0006 to $603,445 an hour, and those extremes do not look like real hourly pay. For each income year we find the 1st and 99th percentile of `hrwage_real`, weighted with `asecwt`, using **all workers** (full-time and part-time, men and women together). Wages below the 1st or above the 99th percentile are dropped. There is one set of cutoffs per year, and every table uses it.

In [ ]:
def cutoffs_for(data):                                     # one year of data
    return pd.Series({"p1": wquantile(data, "hrwage_real", 0.01), "p99": wquantile(data, "hrwage_real", 0.99)})   # low and high cutoff

cutoffs = df.groupby("year").apply(cutoffs_for, include_groups=False)   # one row per year, all workers
print(cutoffs.round(1))                                    # dollars per hour, 2025 dollars

In [ ]:
df["p1"] = df["year"].map(cutoffs["p1"])                   # each row gets its year's low cutoff
df["p99"] = df["year"].map(cutoffs["p99"])                 # and its year's high cutoff
df["keep"] = (df["hrwage_real"] >= df["p1"]) & (df["hrwage_real"] <= df["p99"])   # True = inside the cutoffs
df["trimmed"] = (~df["keep"]).astype(int)                  # 1 = dropped by the trim

kept = df[df["keep"]]                                      # workers inside the cutoffs
kept_ft = kept[kept["fulltime"] == 1]                      # full-time workers inside the cutoffs

Check: how much did the trim drop? For all workers, each end drops about 1%, so about 2% in total. For full-time workers the share can differ, because the cutoffs come from everyone.

In [ ]:
check = pd.DataFrame({
    "all workers": df.groupby("year").apply(wmean, "trimmed", include_groups=False) * 100,                          # % of people dropped
    "full-time": df[df["fulltime"] == 1].groupby("year").apply(wmean, "trimmed", include_groups=False) * 100,       # % of full-time workers dropped
})
print(check.round(1))

ft_total = (df["fulltime"] == 1).sum()                     # full-time workers in the file (unweighted rows)
ft_dropped = df.loc[df["fulltime"] == 1, "trimmed"].sum()  # of those, dropped by the trim
print("Rows dropped by the trim:", df["trimmed"].sum(), "of", len(df))
print("Full-time rows dropped by the trim:", ft_dropped, "of", ft_total)

---
## 4. Table 1: hourly wage by sex
**Question.** How much do women earn per hour compared with men, year by year?

**Who is in it?** Full-time workers inside the trim cutoffs. **What is in each cell?** Dollars per hour in 2025 dollars, weighted with `asecwt`. The main gap uses means: men minus women in dollars, and women's wage as a % of men's. The medians are there for reference. Rows are income years. Under the table, the unweighted number of people (n) behind each year is a data check.

In [ ]:
sex_order = ["Male", "Female"]                                                       # our column order
mean_wage = kept_ft.groupby(["year", "female_label"]).apply(wmean, "hrwage_real", include_groups=False)   # weighted mean
mean_wage = mean_wage.unstack()[sex_order]                                           # years down the rows, sex across
median_wage = kept_ft.groupby(["year", "female_label"]).apply(wquantile, "hrwage_real", 0.5, include_groups=False)   # weighted median
median_wage = median_wage.unstack()[sex_order]                                       # same layout

table1 = pd.DataFrame({
    "Mean: Men": mean_wage["Male"],                                                  # dollars per hour
    "Mean: Women": mean_wage["Female"],
    "Mean: Men minus women ($)": mean_wage["Male"] - mean_wage["Female"],            # gap in dollars
    "Mean: Women as % of men": mean_wage["Female"] / mean_wage["Male"] * 100,        # gap in percent
    "Median: Men": median_wage["Male"],
    "Median: Women": median_wage["Female"],
    "Median: Women as % of men": median_wage["Female"] / median_wage["Male"] * 100,  # for reference
})
print("Table 1. Hourly wage of full-time workers by sex, 2025 dollars")
print(table1.round(1))

n_people = kept_ft.groupby(["year", "female_label"]).size().unstack()[sex_order]     # unweighted rows: a data check
print("Unweighted number of people behind Table 1")
print(n_people.rename(columns={"Male": "Men (n)", "Female": "Women (n)"}))
print("Rows left out: no missing wage or sex;", ft_dropped, "of", ft_total, "full-time rows dropped by the trim")

In [ ]:
fig1_path = "/content/drive/My Drive/fig1_gap_ratio.png"                       # where to save the chart
fig1_table = table1[["Mean: Women as % of men", "Median: Women as % of men"]].round(1)   # the two gap columns of Table 1
fig1_table.columns = ["Mean", "Median"]                                        # short names for the legend
print("Figure 1 data: the two 'Women as % of men' columns of Table 1, full-time workers")
print(fig1_table)

fig, ax = plt.subplots(figsize=(9, 5))                                         # one chart
ax.plot(fig1_table.index, fig1_table["Mean"], color="#2a78d6", linewidth=2, label="Mean")                       # solid blue line
ax.plot(fig1_table.index, fig1_table["Median"], color="#eb6834", linewidth=2, linestyle="--", label="Median")   # dashed orange line
ax.set_ylim(0, 100)                                                            # y-axis from 0 to 100
ax.set_yticks(range(0, 101, 20))                                               # y ticks every 20
ax.set_xlim(1975, 2026)                                                        # a little room on both sides
ax.set_xticks([1976] + list(range(1980, 2026, 5)))                             # x ticks: first year, then every 5 years
ax.tick_params(axis="both", direction="out", length=5)                         # tick marks on both axes
ax.set_ylabel("Women's hourly wage as % of men's")                             # units
ax.set_xlabel("Income year")
ax.set_title("Women's hourly wage as % of men's, full-time workers, 1976-2025", loc="left")
ax.text(0, 1.13, "Figure 1", transform=ax.transAxes, fontweight="bold")       # figure label above the title
ax.grid(axis="y", color="#e5e5e3", linewidth=0.8)                              # light horizontal lines
ax.set_axisbelow(True)                                                         # grid behind the lines
ax.spines[["top", "right"]].set_visible(False)                                 # cleaner frame
ax.legend(frameon=False, loc="lower right")                                    # which line is which
fig.text(0.01, -0.02, "Source: CPS ASEC 1976-2025 via IPUMS, full-time workers ages 16-64, weighted with asecwt.", ha="left", va="top", fontsize=9)   # source note at the bottom
fig.savefig(fig1_path, dpi=200, bbox_inches="tight")                           # save before show
plt.show()
print("Saved to My Drive")

## Write-up (50-100 words)

Question:

Finding:

---
## 5. Table 2: full-time, part-time and all workers
**Question.** Does the gap differ between full-time and part-time workers?

**Who is in it?** Workers inside the trim cutoffs: full-time (`kept_ft`), part-time (`kept` with `fulltime == 0`, under 35 hours), and all workers (`kept`). **What is in each cell?** The gap columns are women's mean hourly wage as a % of men's, in 2025 dollars, weighted with `asecwt`. "Full-time minus part-time" is the difference in percentage points. The part-time columns are the weighted share of women, and of men, who work part-time. Each share is out of all women (or all men) in that year, so it adds up to 100 with the full-time share. Under the table, the unweighted number of people (n) in each group and year is a data check, with the smallest count.

In [ ]:
kept_pt = kept[kept["fulltime"] == 0]                                          # part-time workers inside the cutoffs
mean_pt = kept_pt.groupby(["year", "female_label"]).apply(wmean, "hrwage_real", include_groups=False)   # weighted mean, part-time
mean_pt = mean_pt.unstack()[sex_order]                                         # years down the rows, sex across
mean_all = kept.groupby(["year", "female_label"]).apply(wmean, "hrwage_real", include_groups=False)     # weighted mean, all workers
mean_all = mean_all.unstack()[sex_order]                                       # same layout
gap_ft = table1["Mean: Women as % of men"]                                     # full-time gap, from Table 1 (means)
gap_pt = mean_pt["Female"] / mean_pt["Male"] * 100                             # same gap, part-time
gap_all = mean_all["Female"] / mean_all["Male"] * 100                          # same gap, all workers

shares = pd.crosstab([kept["year"], kept["female_label"]], kept["fulltime_label"],
                     values=kept["asecwt"], aggfunc="sum", normalize="index") * 100   # each row (year and sex) adds up to 100
parttime = shares["Part-time"].unstack()[sex_order]                            # % of each sex who work part-time

table2 = pd.DataFrame({
    "Full-time: Women as % of men": gap_ft,
    "Part-time: Women as % of men": gap_pt,
    "All workers: Women as % of men": gap_all,
    "Full-time minus part-time (pp)": gap_ft - gap_pt,                         # percentage points
    "Women working part-time (%)": parttime["Female"],
    "Men working part-time (%)": parttime["Male"],
})
print("Table 2. Gap for full-time, part-time and all workers, 2025 dollars")
print(table2.round(1))

In [ ]:
n_ft = kept_ft.groupby(["year", "female_label"]).size().unstack()[sex_order]    # unweighted rows, full-time
n_pt = kept_pt.groupby(["year", "female_label"]).size().unstack()[sex_order]   # unweighted rows, part-time
n_all = kept.groupby(["year", "female_label"]).size().unstack()[sex_order]     # unweighted rows, all workers
n_table2 = pd.DataFrame({
    "Full-time: Men (n)": n_ft["Male"], "Full-time: Women (n)": n_ft["Female"],
    "Part-time: Men (n)": n_pt["Male"], "Part-time: Women (n)": n_pt["Female"],
    "All workers: Men (n)": n_all["Male"], "All workers: Women (n)": n_all["Female"],
})
print("Unweighted number of people behind Table 2")
print(n_table2)
smallest = n_table2.stack().idxmin()                                           # (year, column) of the smallest count
print("Smallest count in any cell:", n_table2.stack().min(), "- year", smallest[0], "-", smallest[1])
missing = df[["hrwage_real", "female", "fulltime"]].isna().any(axis=1).sum()   # rows missing wage, sex or full-time status
print("Rows left out:", missing, "missing wage, sex or full-time status;", df["trimmed"].sum(), "of", len(df), "rows dropped by the trim")

In [ ]:
fig2_path = "/content/drive/My Drive/fig2_work_status.png"                     # where to save the figure
old_fig2_path = "/content/drive/My Drive/fig2_fulltime_vs_all.png"             # the old Figure 2 file, replaced by this one
gap_columns = ["Full-time: Women as % of men", "Part-time: Women as % of men", "All workers: Women as % of men"]   # left panel
share_columns = ["Women working part-time (%)", "Men working part-time (%)"]  # right panel
fig2_table = table2[gap_columns + share_columns].round(1)                      # the five columns the figure shows
print("Table 2 columns shown in Figure 2")
print(fig2_table)

y_top = np.ceil((max(100, fig2_table[gap_columns].max().max()) + 5) / 20) * 20   # high enough for every value and the 100 line
fig2, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(14, 5))                # two panels side by side
ax_left.plot(fig2_table.index, fig2_table[gap_columns[0]], color="#2a78d6", linewidth=2, label=gap_columns[0])                    # solid blue
ax_left.plot(fig2_table.index, fig2_table[gap_columns[1]], color="#1baf7a", linewidth=2, linestyle="-.", label=gap_columns[1])    # dash-dot aqua
ax_left.plot(fig2_table.index, fig2_table[gap_columns[2]], color="#eb6834", linewidth=2, linestyle="--", label=gap_columns[2])    # dashed orange
ax_left.axhline(100, color="#52514e", linestyle="--", linewidth=1)             # thin dashed line at 100: equal pay
ax_left.text(2025.8, 101.5, "No gap", ha="right", va="bottom", color="#52514e")   # label for the line at 100
ax_right.plot(fig2_table.index, fig2_table[share_columns[0]], color="#6250d6", linewidth=2, label=share_columns[0])                # solid violet
ax_right.plot(fig2_table.index, fig2_table[share_columns[1]], color="#e87ba4", linewidth=2, linestyle="--", label=share_columns[1])   # dashed magenta
ax_left.set_ylim(0, y_top)                                                     # starts at 0, high enough for every value
ax_left.set_yticks(range(0, int(y_top) + 1, 20))                               # y ticks every 20
ax_right.set_ylim(0, 100)                                                      # shares: 0 to 100
ax_right.set_yticks(range(0, 101, 20))                                         # y ticks every 20
ax_left.set_ylabel("Women's hourly wage as % of men's")                        # units
ax_right.set_ylabel("% working part-time")
ax_left.set_title("Gap by work status", loc="left")
ax_right.set_title("Share working part-time", loc="left")
ax_left.legend(frameon=False, loc="lower right")                               # lines sit high, so the legend goes low
ax_right.legend(frameon=False, loc="upper right")                              # lines sit low, so the legend goes high
for ax in (ax_left, ax_right):                                                 # settings both panels share
    ax.set_xlim(1975, 2026)                                                    # a little room on both sides
    ax.set_xticks([1976] + list(range(1980, 2026, 5)))                         # x ticks: first year, then every 5 years
    ax.set_xlabel("Income year")
    ax.tick_params(axis="both", direction="out", length=5)                     # tick marks on both axes
    ax.grid(axis="y", color="#e5e5e3", linewidth=0.8)                          # light horizontal lines
    ax.set_axisbelow(True)                                                     # grid behind the lines
    ax.spines[["top", "right"]].set_visible(False)                             # cleaner frame
fig2.suptitle("Figure 2. Gap for full-time, part-time and all workers, 1976-2025", x=0.01, ha="left", fontsize=14)   # overall title
fig2.tight_layout(rect=(0, 0, 1, 0.97))                                        # leave room for the overall title
fig2.text(0.01, -0.02, "Source: CPS ASEC 1976-2025 via IPUMS, ages 16-64, weighted with asecwt.", ha="left", va="top", fontsize=9)   # source note at the bottom
fig2.savefig(fig2_path, dpi=200, bbox_inches="tight")                          # save before show
if os.path.exists(old_fig2_path):                                              # the old Figure 2 file is still on Drive
    os.remove(old_fig2_path)                                                   # replace it: remove the old file
    print("Removed the old file:", old_fig2_path)
plt.show()
print("Saved to My Drive")

## Write-up (50-100 words)

Question:

Finding:

---
## 6. Table 3: gap by age group
**Question.** Is the gap different for younger and older women?

**Who is in it?** Full-time workers inside the trim cutoffs (`kept_ft`), in five age groups. **What is in each cell?** Women's weighted mean hourly wage as a % of men's, in 2025 dollars, weighted with `asecwt`, for one year and one age group. Rows with no age group are left out. The last column, All ages, is the gap for all full-time workers (the mean gap from Table 1), for comparison. Under the table, the unweighted number of people (n) in each year and group is a data check, with the smallest count.

**A new tool: `gap_by`.** `gap_by(group_col, order)` builds this table for any group column. It returns two tables: the gap (years down the rows, groups across, in the order we give) and the unweighted number of men and women in each year and group. We use it again for the other group tables.

In [ ]:
def gap_by(group_col, order):                                                   # a label column and the order of its groups
    data = kept_ft[kept_ft[group_col].notna()]                                  # full-time workers inside the cutoffs; rows with no group are left out
    means = data.groupby(["year", group_col, "female_label"]).apply(wmean, "hrwage_real", include_groups=False)   # weighted mean for each year, group, sex
    means = means.unstack("female_label")                                       # sex across: Female and Male columns
    gap = (means["Female"] / means["Male"] * 100).unstack(group_col)[order]     # women as % of men: years down, groups across, our order
    counts = pd.crosstab(data["year"], [data[group_col], data["female_label"]])  # unweighted people: years down, (group, sex) across
    counts = counts.reindex(columns=pd.MultiIndex.from_product([order, ["Male", "Female"]])).fillna(0).astype(int)   # our order, men first
    counts = counts.rename(columns={"Male": "Men", "Female": "Women"}, level=1)  # names for printing
    return gap, counts

In [ ]:
age_order = ["16-24", "25-34", "35-44", "45-54", "55-64"]                      # age groups, youngest first
gap_age, n_age = gap_by("agegrp_label", age_order)                              # gap table and unweighted counts
gap_age["All ages"] = table1["Mean: Women as % of men"]                        # all full-time workers, from Table 1 (means)
print("Table 3. Women's hourly wage as % of men's by age group, full-time workers")
print(gap_age.round(1))

In [ ]:
print("Unweighted number of people behind Table 3")
print(n_age)
smallest_group = n_age.min().idxmin()                                           # (group, sex) column with the smallest count
smallest_year = n_age[smallest_group].idxmin()                                  # the year it happens
print("Smallest count in any cell:", n_age.min().min(), "- year", smallest_year, "-", smallest_group[0], smallest_group[1])
print("Rows left out:", kept_ft["agegrp_label"].isna().sum(), "of", len(kept_ft), "full-time rows have no age group")

In [ ]:
fig3_path = "/content/drive/My Drive/fig3_gap_by_age.png"                      # where to save the figure
fig3_table = gap_age.round(1)                                                  # Table 3, with the All ages column
print("Table 3 as plotted in Figure 3")
print(fig3_table)

y_top = np.ceil((max(100, fig3_table.max().max()) + 5) / 20) * 20              # high enough for every value and the 100 line
fig3, axes = plt.subplots(1, 5, figsize=(20, 4.8), sharey=True)                # five panels side by side, same y-axis
for ax, group in zip(axes, age_order):                                         # one panel per age group, in the table's order
    ax.plot(fig3_table.index, fig3_table["All ages"], color="#8c8c88", linewidth=2.5, label="All ages")   # gray: all full-time workers
    ax.plot(fig3_table.index, fig3_table[group], color="#2a78d6", linewidth=2, label="Age group")         # blue: this panel's age group
    ax.axhline(100, color="#52514e", linestyle="--", linewidth=1)              # dashed line at 100: equal pay
    ax.set_title(group, loc="left")                                            # panel title is the age group
    ax.set_xlabel("Income year")
    ax.set_xlim(1975, 2026)                                                    # a little room on both sides
    ax.set_xticks([1976] + list(range(1985, 2026, 10)))                        # x ticks: first year, then every 10 years
    ax.tick_params(axis="both", direction="out", length=5)                     # tick marks on both axes
    ax.grid(axis="y", color="#e5e5e3", linewidth=0.8)                          # light horizontal lines
    ax.set_axisbelow(True)                                                     # grid behind the lines
    ax.spines[["top", "right"]].set_visible(False)                             # cleaner frame
axes[0].set_ylim(0, y_top)                                                     # shared y-axis: starts at 0, high enough for every value
axes[0].set_yticks(range(0, int(y_top) + 1, 20))                               # y ticks every 20
axes[0].set_ylabel("Women's hourly wage as % of men's")                        # first panel only
handles, labels = axes[0].get_legend_handles_labels()                          # the two lines of the first panel
fig3.legend(handles[::-1], labels[::-1], loc="upper right", ncol=2, frameon=False)   # one legend for the whole figure
fig3.suptitle("Figure 3. Gap by age group, full-time workers, 1976-2025", x=0.01, ha="left", fontsize=14)   # overall title
fig3.tight_layout(rect=(0, 0, 1, 0.92))                                        # leave room for the title and legend
fig3.text(0.01, -0.02, "Source: CPS ASEC 1976-2025 via IPUMS, full-time workers ages 16-64, weighted with asecwt.", ha="left", va="top", fontsize=9)   # source note at the bottom
fig3.savefig(fig3_path, dpi=200, bbox_inches="tight")                          # save before show
plt.show()
print("Saved to My Drive")

## Write-up (50-100 words)

Question:

Finding:

---
## 7. Table 4: gap by education
**Question.** Does the gap differ by how much education workers have?

**Who is in it?** Full-time workers inside the trim cutoffs (`kept_ft`), in five education groups. **What is in each cell?** Women's weighted mean hourly wage as a % of men's, in 2025 dollars, weighted with `asecwt`, for one year and one education group. Rows with no education group are left out. The last column, All education, is the gap for all full-time workers (the mean gap from Table 1), for comparison. Some groups, like None/preschool, may have few people in some years. Under the table, the unweighted number of people (n) in each year and group is a data check, with the smallest count of women and of men in each group across all years.

In [ ]:
educ_order = ["None/preschool", "Less than HS", "High school", "Some college", "Bachelor's+"]   # education groups, lowest first
gap_educ, n_educ = gap_by("educ5_label", educ_order)                            # gap table and unweighted counts
gap_educ["All education"] = table1["Mean: Women as % of men"]                   # all full-time workers, from Table 1 (means)
print("Table 4. Women's hourly wage as % of men's by education, full-time workers")
print(gap_educ.round(1))

In [ ]:
print("Unweighted number of people behind Table 4")
print(n_educ)
min_n = n_educ.min().unstack()                                                  # smallest count in each group and sex, across all years
min_year = n_educ.idxmin().unstack()                                            # the year each smallest count happens
smallest_table = pd.DataFrame({
    "Men: smallest n": min_n["Men"], "Men: year": min_year["Men"],
    "Women: smallest n": min_n["Women"], "Women: year": min_year["Women"],
}).loc[educ_order]                                                              # our group order
print("Smallest count of men and of women in each education group, across all years")
print(smallest_table)
print("Rows left out:", kept_ft["educ5_label"].isna().sum(), "of", len(kept_ft), "full-time rows have no education group")

In [ ]:
fig4_path = "/content/drive/My Drive/fig4_gap_by_educ.png"                     # where to save the figure
fig4_table = gap_educ.round(1)                                                 # Table 4, with the All education column
print("Table 4 as plotted in Figure 4")
print(fig4_table)

y_top = np.ceil((max(100, fig4_table.max().max()) + 5) / 20) * 20              # high enough for every value and the 100 line
fig4, axes = plt.subplots(1, 5, figsize=(20, 4.8), sharey=True)                # five panels side by side, same y-axis
for ax, group in zip(axes, educ_order):                                        # one panel per education group, in the table's order
    ax.plot(fig4_table.index, fig4_table["All education"], color="#8c8c88", linewidth=2.5, label="All education")   # gray: all full-time workers
    ax.plot(fig4_table.index, fig4_table[group], color="#2a78d6", linewidth=2, label="Education group")             # blue: this panel's group
    ax.axhline(100, color="#52514e", linestyle="--", linewidth=1)              # dashed line at 100: equal pay
    ax.set_title(group, loc="left")                                            # panel title is the education group
    ax.set_xlabel("Income year")
    ax.set_xlim(1975, 2026)                                                    # a little room on both sides
    ax.set_xticks([1976] + list(range(1985, 2026, 10)))                        # x ticks: first year, then every 10 years
    ax.tick_params(axis="both", direction="out", length=5)                     # tick marks on both axes
    ax.grid(axis="y", color="#e5e5e3", linewidth=0.8)                          # light horizontal lines
    ax.set_axisbelow(True)                                                     # grid behind the lines
    ax.spines[["top", "right"]].set_visible(False)                             # cleaner frame
axes[0].set_ylim(0, y_top)                                                     # shared y-axis: starts at 0, high enough for every value
axes[0].set_yticks(range(0, int(y_top) + 1, 20))                               # y ticks every 20
axes[0].set_ylabel("Women's hourly wage as % of men's")                        # first panel only
handles, labels = axes[0].get_legend_handles_labels()                          # the two lines of the first panel
fig4.legend(handles[::-1], labels[::-1], loc="upper right", ncol=2, frameon=False)   # one legend for the whole figure
fig4.suptitle("Figure 4. Gap by education, full-time workers, 1976-2025", x=0.01, ha="left", fontsize=14)   # overall title
fig4.tight_layout(rect=(0, 0, 1, 0.92))                                        # leave room for the title and legend
fig4.text(0.01, -0.02, "Source: CPS ASEC 1976-2025 via IPUMS, full-time workers ages 16-64, weighted with asecwt.", ha="left", va="top", fontsize=9)   # source note at the bottom
fig4.savefig(fig4_path, dpi=200, bbox_inches="tight")                          # save before show
plt.show()
print("Saved to My Drive")

## Write-up (50-100 words)

Question:

Finding:

---
## 8. Table 5: gap by race and ethnicity
**Question.** Does the gap differ by race and ethnicity?

**Who is in it?** Full-time workers inside the trim cutoffs (`kept_ft`), in four groups. White and Black mean non-Hispanic. Hispanic means any race. **What is in each cell?** Women's weighted mean hourly wage as a % of men's, in 2025 dollars, weighted with `asecwt`, for one year and one group. Rows with no race or ethnicity (the "don't know" and "no response" answers) are left out of the four groups. The last column, All races, is the gap for all full-time workers (the mean gap from Table 1), so it includes those rows. Under the table, the unweighted number of people (n) in each year and group is a data check, with the smallest count and the number of rows left out.

In [ ]:
race_order = ["White", "Black", "Hispanic", "Other"]                           # our group order
gap_race, n_race = gap_by("raceeth_label", race_order)                          # gap table and unweighted counts
gap_race["All races"] = table1["Mean: Women as % of men"]                       # all full-time workers, from Table 1 (means)
print("Table 5. Women's hourly wage as % of men's by race and ethnicity, full-time workers")
print(gap_race.round(1))

In [ ]:
print("Unweighted number of people behind Table 5")
print(n_race)
smallest_group = n_race.min().idxmin()                                          # (group, sex) column with the smallest count
smallest_year = n_race[smallest_group].idxmin()                                 # the year it happens
print("Smallest count in any cell:", n_race.min().min(), "- year", smallest_year, "-", smallest_group[0], smallest_group[1])
missing_race = kept_ft["raceeth_label"].isna().sum()                            # full-time rows with no race or ethnicity
print("Rows left out:", missing_race, "of", len(kept_ft), "full-time rows have no race or ethnicity (" + str(round(missing_race / len(kept_ft) * 100, 1)) + "%)")

In [ ]:
fig5_path = "/content/drive/My Drive/fig5_gap_by_race.png"                     # where to save the figure
fig5_table = gap_race.round(1)                                                 # Table 5, with the All races column
print("Table 5 as plotted in Figure 5")
print(fig5_table)

y_top = np.ceil((max(100, fig5_table.max().max()) + 5) / 20) * 20              # high enough for every value and the 100 line
race_colors = {"White": "#2a78d6", "Black": "#eb6834", "Hispanic": "#1baf7a", "Other": "#6250d6"}   # one color per group
fig5, ax = plt.subplots(figsize=(10, 5.5))                                     # one chart
for group in race_order:                                                       # one line per group, in the table's order
    style = "-." if group == "Other" else "-"                                  # Other is dash-dot, so it differs from blue without color
    ax.plot(fig5_table.index, fig5_table[group], color=race_colors[group], linewidth=2, linestyle=style, label=group, zorder=3)
ax.plot(fig5_table.index, fig5_table["All races"], color="#8c8c88", linewidth=2.5, linestyle=(0, (5, 3)), label="All races", zorder=2)   # gray dashed: all full-time workers
ax.axhline(100, color="#52514e", linestyle="--", linewidth=1)                  # thin dashed line at 100: equal pay
ax.text(2025.8, 101.5, "No gap", ha="right", va="bottom", color="#52514e")     # label for the line at 100
ax.set_ylim(0, y_top)                                                          # starts at 0, high enough for every value
ax.set_yticks(range(0, int(y_top) + 1, 20))                                    # y ticks every 20
ax.set_xlim(1975, 2026)                                                        # a little room on both sides
ax.set_xticks([1976] + list(range(1980, 2026, 5)))                             # x ticks: first year, then every 5 years
ax.set_ylabel("Women's hourly wage as % of men's")                             # units
ax.set_xlabel("Income year")
ax.tick_params(axis="both", direction="out", length=5)                         # tick marks on both axes
ax.grid(axis="y", color="#e5e5e3", linewidth=0.8)                              # light horizontal lines
ax.set_axisbelow(True)                                                         # grid behind the lines
ax.spines[["top", "right"]].set_visible(False)                                 # cleaner frame
ax.legend(title="Race and ethnicity", frameon=False, loc="lower right")        # lines sit high, so the legend goes low
ax.set_title("Figure 5. Gap by race and ethnicity, full-time workers, 1976-2025", loc="left", fontsize=13)   # title
fig5.text(0.01, -0.02, "Source: CPS ASEC 1976-2025 via IPUMS, full-time workers ages 16-64, weighted with asecwt.", ha="left", va="top", fontsize=9)   # source note at the bottom
fig5.savefig(fig5_path, dpi=200, bbox_inches="tight")                          # save before show
plt.show()
print("Saved to My Drive")

## Write-up (50-100 words)

Question:

Finding:

---
## 9. Table 6: gap by marital status
**Question.** Does the gap differ by marital status?

**Who is in it?** Full-time workers inside the trim cutoffs (`kept_ft`), in three groups. **What is in each cell?** Women's weighted mean hourly wage as a % of men's, in 2025 dollars, weighted with `asecwt`, for one year and one group. Rows with no marital status would be left out. The last column, All, is the gap for all full-time workers (the mean gap from Table 1), for comparison. Under the table, the unweighted number of people (n) in each year and group is a data check, with the smallest count.

In [ ]:
marst_order = ["Married", "Previously married", "Never married"]                # our group order
gap_marst, n_marst = gap_by("marst3_label", marst_order)                        # gap table and unweighted counts
gap_marst["All"] = table1["Mean: Women as % of men"]                            # all full-time workers, from Table 1 (means)
print("Table 6. Women's hourly wage as % of men's by marital status, full-time workers")
print(gap_marst.round(1))

In [ ]:
print("Unweighted number of people behind Table 6")
print(n_marst)
smallest_group = n_marst.min().idxmin()                                         # (group, sex) column with the smallest count
smallest_year = n_marst[smallest_group].idxmin()                                # the year it happens
print("Smallest count in any cell:", n_marst.min().min(), "- year", smallest_year, "-", smallest_group[0], smallest_group[1])
print("Rows left out:", kept_ft["marst3_label"].isna().sum(), "of", len(kept_ft), "full-time rows have no marital status")